# HW-2: полный инференс на Kaggle

Ноутбук скачивает `test.parquet`, загружает текущую модель с RoPE/RMSNorm/SwiGLU и общий токенизатор, генерирует решения для всех 1919 задач в исходном порядке и сохраняет `/kaggle/working/predictions.jsonl`. Параметры генерации: beam search 3, batch size 32 и до 256 новых токенов.

In [ ]:
%pip install -q tokenizers

In [ ]:
import gc
import json
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd
import torch
from tokenizers import Tokenizer

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/predictions.jsonl')
EXPECTED_COUNT = 1919
BATCH_SIZE = 32
MAX_NEW_TOKENS = 256
NUM_BEAMS = 3
LENGTH_PENALTY = 1.0
TEST_URL = 'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/main/data/test/test.parquet?download=true'

assert torch.cuda.is_available(), 'В настройках Notebook включите GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
checkpoint_path = find_input(
    'inference-best.pt', 'inference.pt', 'best.pt', 'model.pt'
)
questions_path = Path('/kaggle/working/test.parquet')
urllib.request.urlretrieve(TEST_URL, questions_path)
shared_tokenizer_path = find_input('shared_tokenizer.json')
code_dir = find_input('inference.py').parent
sys.path.insert(0, str(code_dir))

from inference import generate_texts, load_model

print('Checkpoint:', checkpoint_path)
print('Questions:', questions_path)
print('Code:', code_dir)

In [ ]:
test = pd.read_parquet(questions_path)
prompt_column = next(
    (name for name in ('question', 'prompt', 'text') if name in test.columns),
    None,
)
if prompt_column is None:
    raise KeyError(f'Не найдена колонка с условием; колонки: {list(test.columns)}')
prompts = test[prompt_column].astype(str).tolist()

assert len(prompts) == EXPECTED_COUNT, (len(prompts), EXPECTED_COUNT)
print('Загружено задач:', len(prompts))

In [ ]:
device = torch.device('cuda')
shared_tokenizer = Tokenizer.from_file(str(shared_tokenizer_path))
source_tokenizer = shared_tokenizer
target_tokenizer = shared_tokenizer
model = load_model(checkpoint_path, device)
print(f'Параметров: {sum(p.numel() for p in model.parameters()):,}')
print('max_source_length:', model.config.max_source_length)
print('max_new_tokens:', MAX_NEW_TOKENS)
print('num_beams:', NUM_BEAMS)

In [ ]:
started = time.perf_counter()
batch_size = BATCH_SIZE

while True:
    try:
        predictions = generate_texts(
            model,
            source_tokenizer,
            target_tokenizer,
            prompts,
            batch_size=batch_size,
            max_source_length=model.config.max_source_length,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
        )
        break
    except torch.cuda.OutOfMemoryError:
        if batch_size == 1:
            raise
        batch_size = max(1, batch_size // 2)
        gc.collect()
        torch.cuda.empty_cache()
        print('Недостаточно GPU-памяти, повторяем с batch_size =', batch_size)

elapsed = time.perf_counter() - started
print(f'Готово за {elapsed / 60:.1f} мин.; batch_size={batch_size}')

In [ ]:
assert len(predictions) == EXPECTED_COUNT
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    for prediction in predictions:
        file.write(json.dumps({'code': prediction}, ensure_ascii=False) + '\n')

with OUTPUT_PATH.open(encoding='utf-8') as file:
    result = [json.loads(line) for line in file]

assert len(result) == EXPECTED_COUNT
assert all(set(row) == {'code'} and isinstance(row['code'], str) for row in result)
print('Сохранено:', OUTPUT_PATH)
print('Строк:', len(result))
result[:3]

In [ ]:
from IPython.display import FileLink
FileLink(str(OUTPUT_PATH))